# Swiggy Annual Report — RAG Question Answering System

### Retrieval-Augmented Generation (RAG) using LangChain, Groq LLM, and FAISS

This project implements a **Retrieval-Augmented Generation (RAG)** based **Question Answering system** that enables users to query the **Swiggy Annual Report** using natural language.

The system retrieves the **most relevant information from the document** and generates answers **strictly grounded in the report content**, preventing hallucinations.



---

## Install dependencies

In [5]:
# Install all required dependencies for building the RAG pipeline.

!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers faiss-cpu pypdf pillow pytesseract pymupdf

## Imports

In [6]:
# Standard Python libraries for system operations, text processing, and handling binary data
import os
import re
import io

# PyMuPDF (fitz) is used to open and process PDF files
import fitz

# PIL is used to handle images extracted from the PDF
from PIL import Image

# LangChain document schema used to store extracted text along with metadata
from langchain_core.documents import Document

# Text splitter used to divide large documents into smaller chunks for embedding
from langchain_text_splitters import RecursiveCharacterTextSplitter

# FAISS is used as the vector database for storing embeddings and enabling similarity search
from langchain_community.vectorstores import FAISS

# HuggingFace embedding model used to convert text chunks into vector representations
from langchain_huggingface import HuggingFaceEmbeddings

# Prompt template used to structure the prompt sent to the language model
from langchain_core.prompts import ChatPromptTemplate

# Parser used to convert the LLM response into a clean string output
from langchain_core.output_parsers import StrOutputParser

# Runnable utility used to pass the user query directly into the RAG chain
from langchain_core.runnables import RunnablePassthrough

# Groq LLM integration used to access high-performance Llama models
from langchain_groq import ChatGroq

## Upload File

In [7]:
# Import the file upload utility from Google Colab

from google.colab import files

print("Upload the Swiggy Annual Report PDF")
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]

print("Loaded file:", PDF_PATH)

Upload the Swiggy Annual Report PDF


Saving swiggy_annual_report.pdf to swiggy_annual_report (2).pdf
Loaded file: swiggy_annual_report (2).pdf


## PDF Processing

In [8]:
# Function to extract text content from each page of the PDF
def extract_pdf_content(pdf_path):

    # Open the PDF file using PyMuPDF
    doc = fitz.open(pdf_path)

    # List to store extracted documents
    documents = []

    # Iterate through all pages in the PDF
    for page_num, page in enumerate(doc):

        # Extract plain text from the current page
        text = page.get_text("text")

        # If text exists, clean and store it
        if text:

            text = re.sub(r'\s+', ' ', text).strip()

            # Create a LangChain Document object with metadata
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_num + 1,
                        "type": "text",
                        "source": pdf_path
                    }
                )
            )

    # Close the PDF after processing
    doc.close()

    # Return the extracted documents
    return documents


# Run the extraction function on the uploaded PDF
docs = extract_pdf_content(PDF_PATH)

# Print how many document pages were extracted
print(f"Extracted {len(docs)} raw documents")

Extracted 32 raw documents


## Smart Chunking


In [9]:
# Split the extracted documents into smaller chunks for embedding

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)
chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 89


## Embeddings

In [10]:
# Initialize the embedding model used to convert text chunks into vector representations

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Vector Database

In [11]:
# Create a FAISS vector store from the document chunks and their embeddings
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

# Save the vector database locally for reuse
vectorstore.save_local("swiggy_vector_db")

print("Vector database created")

Vector database created


## Retriever

In [12]:
# Configure the retriever to perform similarity search

# It will return the top 5 most relevant chunks for a given query
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

## RAG Prompt

In [13]:
PROMPT = """
You are an expert financial analyst.

Answer the question ONLY using the context below from the Swiggy Annual Report.

Rules:
- Do NOT use external knowledge
- If the answer is not present say:
"I couldn't find this in the Swiggy Annual Report."

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

prompt = ChatPromptTemplate.from_template(PROMPT)

## LLM (Groq Llama3)

In [14]:
from getpass import getpass

## Enter the Groq API key
os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ API key: ")

# Initialize the Groq language model
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

Enter your GROQ API key: ··········


## Format retrieved docs

In [15]:
def format_docs(docs):
    formatted = []

    for d in docs:
        page = d.metadata.get("page","?")
        formatted.append(f"[Page {page}]\n{d.page_content}")

    return "\n\n".join(formatted)

## Full RAG Chain

In [16]:
rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready")

RAG pipeline ready


## Question Answering Interface

In [ ]:
while True:

    query = input("\nAsk a question about Swiggy Annual Report: ")

    if query.lower() == "exit":
        break

    answer = rag_chain.invoke(query)

    print("\nAnswer:\n")
    print(answer)